In [2]:
import numpy as np
import time
import matplotlib.pyplot as plt

# --- Deine Funktionen ---
def proximity_max_freq(
    x,
    sharpness_base=10.0,
    sharpness_growth=2.0,
    decay=0.1,
    max_freq=4,
    weight_exponent=1.0,
    freq_sharpness_exp=1.0,
    norm_x=1.0,
    output_max=1.0
):
    x = np.array(x)
    freqs = np.arange(1, max_freq + 1)
    all_freq_values = []

    for freq in freqs:
        weight = 1.0 / (freq ** weight_exponent)
        freq_sharpness_base = sharpness_base * (freq ** freq_sharpness_exp)
        freq_sharpness_growth = sharpness_growth * (freq ** freq_sharpness_exp)

        sharpness = freq_sharpness_base * (x ** freq_sharpness_growth)
        cos_term = ((1 + np.cos(2 * np.pi * freq * x)) / 2) ** sharpness

        peak_sharpness = freq_sharpness_base * (norm_x ** freq_sharpness_growth)
        peak_value = ((1 + np.cos(2 * np.pi * freq * norm_x)) / 2) ** peak_sharpness
        peak_value = np.where(peak_value == 0, 1e-12, peak_value)

        denom = np.where(x == 0, 1e-12, x ** decay)
        val = weight * (cos_term / peak_value) / denom
        all_freq_values.append(val)

    all_freq_values = np.array(all_freq_values)
    proximity_max = np.max(all_freq_values, axis=0)

    max_at_norm_x = np.max([
        (1.0 / (freq ** weight_exponent)) * 1.0 / (norm_x ** decay)
        for freq in freqs
    ])
    proximity_max = proximity_max / max_at_norm_x * output_max
    return proximity_max


def proximity_max_freq_gaussian(
    x,
    sharpness_base=10.0,
    sharpness_growth=1.0,
    decay=0.1,
    max_freq=4,
    weight_exponent=1.0,
    freq_sharpness_exp=1.0,
    neighbor_peaks=2,
    x_min=0.5,
    x_max=5.0,
    output_max=1.0
):
    """
    Gaussian-based proximity function with true neighbor-peak optimization.
    - Precomputes all relevant (k/f) peaks once
    - For each x, only evaluates the closest `neighbor_peaks`
    Complexity: O(N * neighbor_peaks)
    """
    x = np.array(x, dtype=float)

    # --- 1) Precompute all possible peaks (mu = k/f) in range ---
    peak_list = []
    for f in range(1, max_freq + 1):
        k_max = int(np.ceil(x_max * f))
        for k in range(1, k_max + 1):
            mu = k / f
            if mu < x_min or mu > x_max:
                continue
            weight = 1.0 / (f ** weight_exponent)
            sharpness = sharpness_base * (f ** freq_sharpness_exp) * (k ** sharpness_growth)
            sigma = 1.0 / (sharpness + 1e-9)
            mu_decay = 1.0 / (mu ** decay)
            peak_list.append((mu, sigma, weight * mu_decay))

    peak_list = np.array(peak_list, dtype=float)  # (N, 3): (mu, sigma, amplitude)
    mus, sigmas, amps = peak_list[:, 0], peak_list[:, 1], peak_list[:, 2]

    # --- 2) For each x, evaluate only closest peaks ---
    result = np.zeros_like(x, dtype=float)
    for i, xi in enumerate(x):
        # Find nearest peaks
        idx = np.argpartition(np.abs(mus - xi), neighbor_peaks)[:neighbor_peaks]
        g = np.exp(-0.5 * ((xi - mus[idx]) / sigmas[idx]) ** 2)
        result[i] = np.max(amps[idx] * g)

    # --- 3) Normalize ---
    result /= np.max(result) + 1e-9
    return result * output_max


# --- Benchmark-Skript ---
def benchmark(n_points=10000, repeats=5, seed=42):
    rng = np.random.default_rng(seed)
    x = rng.random(n_points) * 5  # Zufallswerte zwischen 0 und 5

    times = {}
    for name, func in [
        ("cosine", proximity_max_freq),
        ("gaussian", proximity_max_freq_gaussian),
    ]:
        elapsed = []
        for _ in range(repeats):
            start = time.perf_counter()
            func(x)
            end = time.perf_counter()
            elapsed.append(end - start)
        times[name] = (np.mean(elapsed), np.std(elapsed))

    print("Benchmark Ergebnisse (Sekunden):")
    for name, (mean_t, std_t) in times.items():
        print(f"{name:10s}: {mean_t:.6f} ± {std_t:.6f}")

    # Optional: Verhältnis Cos vs Gauss
    speedup = times["gaussian"][0] / times["cosine"][0]
    print(f"\nSpeedup (gaussian / cosine): {speedup:.2f}x")

    return times


if __name__ == "__main__":
    benchmark(n_points=20000, repeats=10)


Benchmark Ergebnisse (Sekunden):
cosine    : 0.016515 ± 0.014650
gaussian  : 0.488040 ± 0.089136

Speedup (gaussian / cosine): 29.55x
